# Citadel Kaggle TPU launcher (thin)
All logic lives in `citadel_tpu/`. This notebook only: env setup → probe → T0 → T1 → receipts. Branch: `citadel`. Cymek is never merged; its SHA is recorded per receipt.

In [ ]:
# 0. Repo: open the pushed citadel branch (or upload this worktree as Kaggle dataset)
!git rev-parse HEAD 2>/dev/null || echo NO_GIT
!git log -1 --oneline 2>/dev/null || true

In [ ]:
# 1. Pin dependencies (TPU-safe; versions recorded into every receipt)
!pip install -q torch torch-xla numpy 2>&1 | tail -2
import torch, platform
try:
    import torch_xla.version as xv; print('torch', torch.__version__, '| xla', getattr(xv, '__version__', 'unknown'), '| py', platform.python_version())
except Exception as e:
    print('XLA_IMPORT_FAIL', repr(e))

In [ ]:
# 2. M0: environment probe (fail-closed; ABORT_NO_TPU on CPU fallback)
from citadel_tpu import environment as env_mod
env = env_mod.main(out='docs/citadel/tpu_receipts/TPU_ENVIRONMENT.json', require_tpu=True)
print(env)

In [ ]:
# 3. T0: single-device one-update certification (MINI_SPEC, bucket 512)
from citadel_tpu import one_update
r0 = one_update.run(out='docs/citadel/tpu_receipts/TPU_ONE_UPDATE.json')
print({k: r0[k] for k in ('certification','loss','tokens_per_second','reload_identical')})

In [ ]:
# 4. Canary data (deterministic; overlap-guarded)
from citadel_tpu import calculator_data as calc
rc = calc.build_all(out_dir='docs/citadel/tpu_receipts/calculator_canary')
print(rc['splits'], rc['split_overlap_rows'])

In [ ]:
# 5. T1: calculator checkpoint (CE only; infrastructure gate, not AGI)
from citadel_tpu import calculator_train
r1 = calculator_train.train(out='docs/citadel/tpu_receipts/TPU_CALCULATOR_CHECKPOINT.json')
print(r1['training'], r1['eval'], r1['interpretation'])

In [ ]:
# 6. Throughput (cold vs steady; steady sizes the 5B plan) + multi-device ledger check
from citadel_tpu import throughput as tp
print(tp.measure_steady_state())
print(tp.certify_multi_device())